In [ ]:
"""
HiGAN+ Batch Inference Script
==============================
Processes all images from input folder and generates handwriting samples
using extracted styles with custom text prompts.
"""

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from pathlib import Path
from tqdm import tqdm
import sys
import os

# ============================================================================
# CONFIGURATION
# ============================================================================

# Paths
PROJECT_PATH = Path(r"B:\College\DL\handwriting_autocomplete_system\Higan+ from Scratch")
INPUT_DIR = PROJECT_PATH / "inference" / "input"
OUTPUT_DIR = PROJECT_PATH / "inference" / "output"
CHECKPOINT_PATH = PROJECT_PATH / "server_files" / "best_e_20.pth"

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# Image processing parameters
IMG_HEIGHT = 64  # Standard height for IAM dataset
MAX_WIDTH = 1600  # Maximum width to handle

# Custom text prompts to generate
CUSTOM_TEXTS = [
    "hello world",
    "deep learning",
    "artificial intelligence",
    "handwriting synthesis",
    "neural networks",
    "bonjour"
]

# ============================================================================
# LOAD PROJECT DEPENDENCIES
# ============================================================================

# Add project path to sys.path
os.chdir(PROJECT_PATH)
if str(PROJECT_PATH) not in sys.path:
    sys.path.insert(0, str(PROJECT_PATH))

# Import project modules (adjust based on your project structure)
try:
    from lib.utils import yaml2config
    from lib.alphabet import strLabelConverter, get_true_alphabet
    
    from networks.BigGAN_networks import Generator
    from networks.module import StyleEncoder, StyleBackbone
    
    print("✓ Project modules loaded successfully")
except ImportError as e:
    print(f"⚠ Error importing project modules: {e}")
    print("Make sure the project structure is correct and all modules are available")
    sys.exit(1)

# ============================================================================
# LOAD CONFIGURATION
# ============================================================================

config_path = PROJECT_PATH / 'configs' / 'gan_iam.yml'
cfg = yaml2config(str(config_path))
cfg.device = str(DEVICE)

print(f"✓ Configuration loaded from {config_path}")

# ============================================================================
# INITIALIZE LABEL CONVERTER
# ============================================================================

# Get the alphabet key (e.g., 'iam_word') from dataset name
alphabet_key = '_'.join(cfg.dataset.split('_')[:2])
label_converter = strLabelConverter(alphabet_key)
alphabet = label_converter.alphabet
print(f"✓ Label converter initialized (alphabet size: {len(alphabet)})")

# ============================================================================
# INITIALIZE MODELS
# ============================================================================

print("\nInitializing models...")

# Generator
generator = Generator(**cfg.GenModel).to(DEVICE)
print(f"✓ Generator initialized (style_dim: {cfg.GenModel.style_dim})")

# Style Encoder and Backbone
style_backbone = StyleBackbone(**cfg.StyBackbone).to(DEVICE)
style_encoder = StyleEncoder(**cfg.EncModel).to(DEVICE)
print(f"✓ Style encoder initialized")

# ============================================================================
# LOAD CHECKPOINT
# ============================================================================

print(f"\nLoading checkpoint from: {CHECKPOINT_PATH}")

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Checkpoint not found at {CHECKPOINT_PATH}")

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

generator.load_state_dict(checkpoint['generator'])
style_encoder.load_state_dict(checkpoint['style_encoder'])

if 'style_backbone' in checkpoint:
    style_backbone.load_state_dict(checkpoint['style_backbone'])

print(f"✓ Loaded checkpoint (epoch {checkpoint.get('epoch', 'N/A')})")

# Set models to eval mode
generator.eval()
style_encoder.eval()
style_backbone.eval()

# ============================================================================
# IMAGE PREPROCESSING FUNCTIONS
# ============================================================================

def load_and_preprocess_image(image_path, target_height=IMG_HEIGHT, max_width=MAX_WIDTH):
    """
    Load and preprocess an image for style extraction.
    
    Args:
        image_path: Path to input image
        target_height: Target height for resizing
        max_width: Maximum allowed width
    
    Returns:
        Preprocessed tensor and original image dimensions
    """
    # Load image
    img = Image.open(image_path).convert('L')  # Convert to grayscale
    orig_width, orig_height = img.size
    
    # Calculate new dimensions maintaining aspect ratio
    aspect_ratio = orig_width / orig_height
    new_height = target_height
    new_width = int(target_height * aspect_ratio)
    
    if new_width > max_width:
        new_width = max_width
        new_height = int(max_width / aspect_ratio)
    
    # Resize image
    img = img.resize((new_width, new_height), Image.LANCZOS)
    
    # Convert to numpy array and normalize
    img_np = np.array(img).astype(np.float32)
    img_np = img_np / 255.0  # Normalize to [0, 1]

    if img_np.mean() > 0.5:
        img_np = 1.0 - img_np
    
    # Convert to tensor [1, 1, H, W]
    img_tensor = torch.from_numpy(img_np).unsqueeze(0).unsqueeze(0)
    
    return img_tensor, (orig_width, orig_height, new_width, new_height)

def tensor_to_image(tensor):
    """
    Convert tensor to numpy image for saving.
    
    Args:
        tensor: Tensor of shape [1, H, W] or [H, W]
    
    Returns:
        Numpy array in uint8 format
    """
    if tensor.dim() == 3:
        tensor = tensor.squeeze(0)
    
    img_np = tensor.cpu().numpy()
    
    # Invert back (white background, black text)
    img_np = 1.0 - img_np
    
    # Clip and convert to uint8
    img_np = np.clip(img_np * 255, 0, 255).astype(np.uint8)
    
    return img_np

# ============================================================================
# INFERENCE FUNCTION
# ============================================================================

@torch.no_grad()
def run_inference(input_dir, output_dir, custom_texts):
    """
    Run inference on all images in input directory.
    
    Args:
        input_dir: Path to input directory containing style reference images
        output_dir: Path to output directory for generated images
        custom_texts: List of text strings to generate
    """
    # Get all image files
    image_extensions = {'.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.tif'}
    image_files = [
        f for f in input_dir.iterdir() 
        if f.suffix.lower() in image_extensions
    ]
    
    if not image_files:
        print(f"⚠ No images found in {input_dir}")
        return
    
    print(f"\nFound {len(image_files)} images to process")
    print(f"Generating {len(custom_texts)} variations per image")
    print("-" * 60)
    
    # Encode custom texts
    custom_lbs, custom_lb_lens = label_converter.encode(custom_texts)
    custom_lbs = custom_lbs.to(DEVICE)
    custom_lb_lens = custom_lb_lens.to(DEVICE)
    
    # Process each image
    for img_idx, img_path in enumerate(tqdm(image_files, desc="Processing images")):
        try:
            # Load and preprocess image
            img_tensor, (orig_w, orig_h, new_w, new_h) = load_and_preprocess_image(img_path)
            img_tensor = img_tensor.to(DEVICE)
            
            # Create image length tensor
            img_len = torch.tensor([new_w], dtype=torch.long).to(DEVICE)
            
            # Extract style from reference image
            style_vector = style_encoder(img_tensor, img_len, style_backbone, vae_mode=False)
            
            # Generate images with custom texts
            generated_images = []
            
            for text_idx, text in enumerate(custom_texts):
                # Get label and length for this text
                text_lb = custom_lbs[text_idx:text_idx+1]
                text_lb_len = custom_lb_lens[text_idx:text_idx+1]
                
                # Generate image
                fake_img = generator(style_vector, text_lb, text_lb_len)
                generated_images.append(fake_img)
            
            # Stack all generated images
            generated_images = torch.cat(generated_images, dim=0)
            
            # Create visualization grid
            n_texts = len(custom_texts)
            fig, axes = plt.subplots(n_texts + 1, 1, figsize=(12, 2 * (n_texts + 1)))
            
            # Plot reference image
            ref_img = tensor_to_image(img_tensor.squeeze())
            axes[0].imshow(ref_img, cmap='gray')
            axes[0].set_title(f'Reference Image: {img_path.name}', 
                            fontsize=12, fontweight='bold')
            axes[0].axis('off')
            
            # Plot generated images
            for i, text in enumerate(custom_texts):
                gen_img = tensor_to_image(generated_images[i])
                axes[i + 1].imshow(gen_img, cmap='gray')
                axes[i + 1].set_title(f'Generated: "{text}"', fontsize=11)
                axes[i + 1].axis('off')
            
            plt.tight_layout()
            
            # Save combined visualization
            output_path = output_dir / f"{img_path.stem}_combined.png"
            plt.savefig(output_path, dpi=150, bbox_inches='tight')
            plt.close()
            
            # Save individual generated images
            for i, text in enumerate(custom_texts):
                gen_img = tensor_to_image(generated_images[i])
                individual_path = output_dir / f"{img_path.stem}_gen_{i+1}_{text.replace(' ', '_')}.png"
                cv2.imwrite(str(individual_path), gen_img)
            
            print(f"✓ Processed {img_path.name}")
            
        except Exception as e:
            print(f"✗ Error processing {img_path.name}: {e}")
            continue
    
    print("-" * 60)
    print(f"✓ Inference complete! Results saved to {output_dir}")
    print(f"  - {len(image_files)} combined visualizations")
    print(f"  - {len(image_files) * len(custom_texts)} individual generated images")


Using device: cuda
✓ Project modules loaded successfully
✓ Configuration loaded from B:\College\DL\handwriting_autocomplete_system\Higan+ from Scratch\configs\gan_iam.yml
✓ Label converter initialized (alphabet size: 80)

Initializing models...
✓ Generator initialized (style_dim: 32)
✓ Project modules loaded successfully
✓ Configuration loaded from B:\College\DL\handwriting_autocomplete_system\Higan+ from Scratch\configs\gan_iam.yml
✓ Label converter initialized (alphabet size: 80)

Initializing models...
✓ Generator initialized (style_dim: 32)
✓ Style encoder initialized

Loading checkpoint from: B:\College\DL\handwriting_autocomplete_system\Higan+ from Scratch\server_files\best_e_20.pth
✓ Style encoder initialized

Loading checkpoint from: B:\College\DL\handwriting_autocomplete_system\Higan+ from Scratch\server_files\best_e_20.pth
✓ Loaded checkpoint (epoch 20)
HiGAN+ BATCH INFERENCE
Input directory:  B:\College\DL\handwriting_autocomplete_system\Higan+ from Scratch\inference\input
O

Processing images:  50%|█████     | 1/2 [00:01<00:01,  1.52s/it]

✓ Processed image.png


Processing images: 100%|██████████| 2/2 [00:02<00:00,  1.09s/it]

✓ Processed Screenshot 2025-11-14 145016.png
------------------------------------------------------------
✓ Inference complete! Results saved to B:\College\DL\handwriting_autocomplete_system\Higan+ from Scratch\inference\output
  - 2 combined visualizations
  - 12 individual generated images

INFERENCE PIPELINE COMPLETE


In [ ]:

# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    print("="*60)
    print("HiGAN+ BATCH INFERENCE")
    print("="*60)
    print(f"Input directory:  {INPUT_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")
    print(f"Checkpoint:       {CHECKPOINT_PATH}")
    print("="*60)
    
    # Check if input directory exists and has images
    if not INPUT_DIR.exists():
        print(f"\n⚠ Input directory does not exist: {INPUT_DIR}")
        print("Creating directory...")
        INPUT_DIR.mkdir(parents=True, exist_ok=True)
        print(f"✓ Created {INPUT_DIR}")
        print("\nPlease add handwriting images to the input directory and run again.")
    else:
        # Run inference
        run_inference(INPUT_DIR, OUTPUT_DIR, CUSTOM_TEXTS)

    print("\n" + "="*60)
    print("INFERENCE PIPELINE COMPLETE")
    print("="*60)